# Entrenamiento del modelo de EcoResiduos

Antes de empezar:

1. Sube `ecoresiduos_colab.zip` a **Mi unidad** de Google Drive, sin descomprimirlo.
2. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**.
3. Ejecuta los pasos en orden, uno a la vez, esperando a que cada uno termine.

Al final se descargan `ecoresiduos_model.tflite` y `labels.txt`, que van en `app/assets/models/`.

## Paso 1. Cargar el dataset

Pide permiso para leer Drive. Si no encuentra el zip allí, muestra un botón para subirlo desde el computador.
Al terminar lista cuántas fotos hay por categoría.

In [ ]:
import os
from google.colab import drive, files

drive.mount('/content/drive')

zip_en_drive = '/content/drive/MyDrive/ecoresiduos_colab.zip'
if os.path.exists(zip_en_drive):
    zip_dataset = zip_en_drive
else:
    print('No encontré el zip en Drive. Súbelo desde el computador:')
    zip_dataset = next(iter(files.upload()))

!rm -rf /content/trabajo && mkdir /content/trabajo && unzip -q "$zip_dataset" -d /content/trabajo
!for d in /content/trabajo/dataset/*/; do echo "$(basename "$d"): $(ls "$d" | wc -l) fotos"; done

## Paso 2. Revisar la GPU e instalar dependencias

Debe mostrar una GPU. Si dice que no hay, cambia el tipo de entorno (ver arriba) y vuelve al paso 1.

In [ ]:
import tensorflow as tf

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'no hay GPU, cambia el tipo de entorno')

!pip -q install scikit-learn matplotlib

## Paso 3 (opcional). Usar un `train.py` más reciente

Solo si cambiaste `train.py` después de subir el zip. Elige el archivo y reemplaza al que venía en el zip.
Si no aplica, salta al paso 4.

In [ ]:
subido = files.upload()
with open('/content/trabajo/train.py', 'wb') as f:
    f.write(next(iter(subido.values())))
print('train.py reemplazado')

## Paso 4. Entrenar

Tarda unos 20 minutos. Los avisos rojos de `cuda_timer` son normales y se pueden ignorar.
Al final debe aparecer `coincide con Keras en 16/16 imágenes`; si sale una advertencia de que no coincide, no uses el modelo.

In [ ]:
%cd /content/trabajo
!python train.py

## Paso 5. Ver resultados

En la matriz de confusión cada fila es la categoría real y cada columna lo que respondió el modelo.
Los números de la diagonal son aciertos; los de fuera, confusiones. Las 8 filas deben tener números.

In [ ]:
from IPython.display import Image, display

print(open('/content/trabajo/reporte_entrenamiento.txt', encoding='utf-8').read())
display(Image('/content/trabajo/matriz_confusion.png'))

## Paso 6. Descargar el modelo

Copia los dos archivos a `app/assets/models/` y ejecuta `flutter clean` antes de probar la app.

In [ ]:
files.download('/content/trabajo/ecoresiduos_model.tflite')
files.download('/content/trabajo/labels.txt')